In [1]:
import pandas as pd
import torch
import spacy

In [11]:
import json

path = "../../data/bronze/archive/phishing and benign email dataset.jsonl"

df = pd.read_json(path, lines=True)


In [12]:
df

,id,subject,body,intent,technique,target,spoofed_sender,label
0,phish-0001,Urgent: Account Verification Required,Your account will be locked in 24 hours. Verif...,Credential Harvesting,Link Spoofing with Urgency,Banking,security@bankofamerica-alert.com,phishing
1,phish-0002,Invoice #3921 Overdue,Your payment is overdue. Download invoice here...,Malware Delivery,Attachment with Malicious Payload,SMBs / Finance,billing@vendorpay.com,phishing
2,phish-0003,Unusual Activity Detected,Suspicious login from Russia. Reset password n...,Credential Harvesting,Homoglyph Link Spoofing,Google,no-reply@g00gle.com,phishing
3,phish-0004,Tax Refund Processing Error,Your refund is delayed. Submit details at: htt...,Credential Harvesting,Fake Government Alert,Taxpayers,service@irs-notify.org,phishing
4,phish-0005,Your Package Awaits!,Delivery issue detected. Update address via: [...,Malware Delivery,Attachment with Malicious Payload,E-commerce,support@fedex-delivery.com,phishing
...,...,...,...,...,...,...,...,...
195,legit-096,Security Policy Update,Review our updated security policies in your a...,Informational,None,Banking,security@wellsfargo.com,benign
196,legit-097,Event Registration Confirmed,You’re registered for our 6/1 seminar. Details...,Transactional,None,General Public,events@meetup.com,benign
197,legit-098,Backup Completed,Your files were backed up successfully. Check ...,Informational,None,Cloud Storage,support@icloud.com,benign
198,legit-099,Plan Upgrade Confirmation,Your plan was upgraded. Access new features now.,Transactional,None,Software Service,billing@slack.com,benign


In [13]:
def clear_irrelevant_coloums(df, columns):

    if df is None:
        raise ValueError("DataFrame not found!")

    new_df = df.drop(columns=columns)
    
    print("Dropping of irrelevant columns successful")

    return new_df


In [14]:
irrelevant_columns = ['id', 'intent', 'technique', 'target', 'spoofed_sender']

df = clear_irrelevant_coloums(df, irrelevant_columns)

df

Dropping of irrelevant columns successful


,subject,body,label
0,Urgent: Account Verification Required,Your account will be locked in 24 hours. Verif...,phishing
1,Invoice #3921 Overdue,Your payment is overdue. Download invoice here...,phishing
2,Unusual Activity Detected,Suspicious login from Russia. Reset password n...,phishing
3,Tax Refund Processing Error,Your refund is delayed. Submit details at: htt...,phishing
4,Your Package Awaits!,Delivery issue detected. Update address via: [...,phishing
...,...,...,...
195,Security Policy Update,Review our updated security policies in your a...,benign
196,Event Registration Confirmed,You’re registered for our 6/1 seminar. Details...,benign
197,Backup Completed,Your files were backed up successfully. Check ...,benign
198,Plan Upgrade Confirmation,Your plan was upgraded. Access new features now.,benign


In [15]:
def concat_subject2body(df, subject_col, body_col, drop = True):

    if df is None:
        raise ValueError("Dataframe not found!")

    new_df = df.copy()

    new_df['Full Text'] = df[subject_col] + ' \n\n ' + df[body_col]

    if drop:
        new_df = new_df.drop(columns=[subject_col, body_col])

    print("Successfully concatenated subject line to body text")

    return new_df


In [16]:
df = concat_subject2body(df, 'subject', 'body')
print(df.head())

Successfully concatenated subject line to body text
      label                                          Full Text
0  phishing  Urgent: Account Verification Required \n\n You...
1  phishing  Invoice #3921 Overdue \n\n Your payment is ove...
2  phishing  Unusual Activity Detected \n\n Suspicious logi...
3  phishing  Tax Refund Processing Error \n\n Your refund i...
4  phishing  Your Package Awaits! \n\n Delivery issue detec...


In [6]:
import re

def mask_email(text):

    if not isinstance(text, str):
        print("Column is not a string type")
        return df

    pattern = r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}"

    replacement = "[EMAIL_ADDRESS]"

    new_text = re.sub(pattern, replacement, text)

    return new_text

def mask_phone(text):

    if not isinstance(text, str):
        print("Column is not a string type")
        return df

    pattern = r"(?<=\d{3})\d{5}"

    replacement = "[PHONE_NUMBER]"

    new_text = re.sub(pattern, replacement, text)

    return new_text

def mask_url(text):

    if not isinstance(text, str):
        print("Column is not a string type")
        return df

    pattern = r"https?:\/\/\S+"
    replacement = "[URL]"
    
    new_text = re.sub(pattern, replacement, text)

    return new_text 

def apply_masking(df, text_column):

    if df is None:
        raise ValueError("Dataframe not found!")

    if not text_column:
        print("You must pass at least one text column")
        return df

    new_df = df.copy()
    new_df["Masked_Text"] = new_df[text_column].apply(mask_email).apply(mask_phone).apply(mask_url)
    
    return new_df

In [20]:
nlp = spacy.load("en_core_web_sm")

def mask_money(df, text_column):

    PII_ENTITIES = {"MONEY"}

    def mask_text(text):
        if not isinstance(text, str):
            return text

        doc = nlp(text)
        text_list = list(text)

        ents = sorted(doc.ents, key=lambda e: e.start_char, reverse=True)

        for ent in sorted(doc.ents, key=lambda e: e.start_char, reverse=True):
            if ent.label_ in PII_ENTITIES:
                text_list[ent.start_char:ent.end_char] = f"[{ent.label_}]"

        return "".join(text_list)

    df = df.copy()
    df[text_column] = df[text_column].apply(mask_text)
    return df


In [7]:
from presidio_analyzer import AnalyzerEngine, BatchAnalyzerEngine
from presidio_anonymizer import BatchAnonymizerEngine

analyzer = AnalyzerEngine()
batch_analyzer = BatchAnalyzerEngine(analyzer_engine=analyzer)
batch_anonymizer = BatchAnonymizerEngine()

def presidio_masking(df, text_column):

    if df is None:
        raise ValueError("Dataframe not found!")
    
    if not isinstance(text_column, str):
        print("Column must be of string type")
        return df

    texts_dict = {text_column: df[text_column].fillna("").astype(str).tolist()}

    analyzer_results = batch_analyzer.analyze_dict(
        texts_dict,
        language="en",
        entities=["NAME", "EMAIL_ADDRESS", "PHONE_NUMBER", "ORGANIZATION", "LOCATION", "URL", "DATE_TIME", "CREDIT_CARD"],
    )

    anonymizer_results = batch_anonymizer.anonymize_dict(
        analyzer_results
    )

    masked_list = anonymizer_results.get(text_column, [])

    output_df = df.copy()
    output_df[text_column] = masked_list

    print(f"Successfully masked {text_column} column")
    return output_df

In [17]:
text_with_email = "Your account don.matrick@gmail.com has been hacked! Go to this website: http://we-are-malicious-site.com to login to reset password immediately."
text_with_phone = "Our records indicate that you have yet to pay off your student loans. Call us at 096738467 to start the payment process"

new_rows = pd.DataFrame({'label': ["phishing", "benign"], 'Full Text': [text_with_email, text_with_phone]})

df = pd.concat([df, new_rows], ignore_index = True)

In [21]:
df["Masked Text"] = df["Full Text"]

masked_df = presidio_masking(df, "Masked Text")
masked_df = mask_money(masked_df, "Masked Text")

masked_df

Successfully masked Masked Text column


,label,Full Text,Masked Text
0,phishing,Urgent: Account Verification Required \n\n You...,Urgent: Account Verification Required \n\n You...
1,phishing,Invoice #3921 Overdue \n\n Your payment is ove...,Invoice #3921 Overdue \n\n Your payment is ove...
2,phishing,Unusual Activity Detected \n\n Suspicious logi...,Unusual Activity Detected \n\n Suspicious logi...
3,phishing,Tax Refund Processing Error \n\n Your refund i...,Tax Refund Processing Error \n\n Your refund i...
4,phishing,Your Package Awaits! \n\n Delivery issue detec...,Your Package Awaits! \n\n Delivery issue detec...
...,...,...,...
197,benign,Backup Completed \n\n Your files were backed u...,Backup Completed \n\n Your files were backed u...
198,benign,Plan Upgrade Confirmation \n\n Your plan was u...,Plan Upgrade Confirmation \n\n Your plan was u...
199,benign,Customer Survey \n\n Share feedback on your re...,Customer Survey \n\n Share feedback on your re...
200,phishing,Your account don.matrick@gmail.com has been ha...,Your account <EMAIL_ADDRESS> has been hacked! ...


In [34]:
nlp = spacy.load("en_core_web_trf")

column = "Full Text"
masked_column = "Masked_Text"

df = apply_masking(df, column)
df = apply_spacy(df, masked_column)

In [23]:
df

,label,Full Text,Masked_Text
0,phishing,Urgent: Account Verification Required \n\n You...,Urgent: Account Verification Required \n\n You...
1,phishing,Invoice #3921 Overdue \n\n Your payment is ove...,Invoice #3921 Overdue \n\n Your payment is ove...
2,phishing,Unusual Activity Detected \n\n Suspicious logi...,Unusual Activity Detected \n\n Suspicious logi...
3,phishing,Tax Refund Processing Error \n\n Your refund i...,Tax Refund Processing Error \n\n Your refund i...
4,phishing,Your Package Awaits! \n\n Delivery issue detec...,Your Package Awaits! \n\n Delivery issue detec...
...,...,...,...
197,benign,Backup Completed \n\n Your files were backed u...,Backup Completed \n\n Your files were backed u...
198,benign,Plan Upgrade Confirmation \n\n Your plan was u...,[ORG] Your plan was upgraded. Access new featu...
199,benign,Customer Survey \n\n Share feedback on your re...,Customer Survey \n\n Share feedback on your re...
200,phishing,Your account don.matrick@gmail.com has been ha...,Your account [EMAIL_ADDRESS] has been hacked! ...


In [35]:
import os

output_dir = "../../data/preprocessed/silver"
file_name = "example_dataset.csv"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

full_path = os.path.join(output_dir, file_name)
df.to_csv(full_path, index=False)

print("Dataframe successfully save to path")

Dataframe successfully save to path
